# Build the combined Talmi EEG dataset

This notebook is the executable entry point for `data/TalmiEEG.h5`. The implementation and acceptance gates live in `lpp_ecmr.data_preparation` and `lpp_ecmr.data_contract`, where they can also be exercised by pytest.

The output contains mixed, pure-negative, and pure-neutral lists. Category lists use the pure-neutral list code while remaining distinguishable through `cond3 == 3`.

In [ ]:
import json
import os
from pathlib import Path

from jaxcmr.helpers import load_data

from lpp_ecmr.data_contract import (
    MIXED_TRIAL_QUERY,
    validate_combined_dataset,
    validate_legacy_mixed_semantics,
)
from lpp_ecmr.data_preparation import (
    BuildInputs,
    build_combined_dataset,
    read_hdf5_dataset,
    write_build,
)


## Source locations

Set `TALMI_ZARUBIN_ARCHIVE` when the lab volume is mounted elsewhere. Raw participant files remain in the lab archive; the validated HDF5 product is tracked in this project.

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "data":
    project_root = project_root.parent

archive_root = Path(
    os.environ.get(
        "TALMI_ZARUBIN_ARCHIVE",
        "/Volumes/rfs-dt492-talmilab-fs/Talmi lab space/Archive/Secondary_Data_Analysis_Zarubin_From_Robin",
    )
)
behavior_dir = archive_root / "Data/Behaviour/Behaviour_csv_files"
delivery_dir = archive_root / "Final_PureList_Analysis_2026-07-15/Data/Extracted_Behavioural_and_LPP_Data"

inputs = BuildInputs(
    mixed_behavior=behavior_dir / "All_Included_Subjects.csv",
    mixed_eeg=delivery_dir / "Single_Trial_Behavioural_and_EEG_Data_Z.csv",
    pure_behavior_dir=behavior_dir,
    pure_eeg=delivery_dir / "Single_Trial_Behavioural_and_EEG_Data_Z_PureList.csv",
)
inputs


## Construct and validate in memory

Construction fails before writing if the 342 mixed rows differ from the frozen per-field legacy digests or if any combined-data invariant is violated.

In [ ]:
result = build_combined_dataset(inputs)
print(json.dumps(result.metadata["summary"], indent=2))
print(json.dumps(result.metadata["diagnostics"], indent=2))


## Write and round-trip

Correctness is enforced by the Python validation functions and test suite; no sidecar manifest is required.

In [ ]:
output_path = project_root / "data/TalmiEEG.h5"
write_build(
    result,
    output_path,
    force=True,
)

raw_round_trip = read_hdf5_dataset(output_path)
round_trip_summary = validate_combined_dataset(raw_round_trip)
validate_legacy_mixed_semantics(raw_round_trip)

loaded = load_data(output_path)
assert loaded["subject"].shape == (639, 1)
assert loaded["list_type"].shape == (639, 1)
print(MIXED_TRIAL_QUERY)
round_trip_summary
